# Задание 05. Фильтры лекарственноподобности: правило Липински

Проверим набор молекул на правило Липински и PAINS-фрагменты.

## Шаг 1. Набор молекул

Бытовые вещества и «сомнительные» молекулы (попросите агента придумать пару SMILES с большой массой).

In [ ]:
from rdkit import Chem

smiles_list = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",   # аспирин
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O", # ибупрофен
    "CC(=O)Nc1ccc(O)cc1",          # парацетамол
    "CCO",                         # этанол
    # добавьте сюда 2 SMILES от агента
]
mols = [Chem.MolFromSmiles(s) for s in smiles_list]
print("Молекул в наборе:", len([m for m in mols if m]))


## Шаг 2. Свойства для правила Липински

In [ ]:
from rdkit.Chem import Descriptors

for s, m in zip(smiles_list, mols):
    if m is None:
        print(s, "— НЕ читается")
        continue
    print(s, "масса=", round(Descriptors.MolWt(m), 1),
          "logP=", round(Descriptors.MolLogP(m), 2),
          "доноры=", Descriptors.NumHDonors(m),
          "акцепторы=", Descriptors.NumHAcceptors(m))


## Шаг 3. Функция «проходит Липински»

In [ ]:
def lipinski_ok(m):
    return (Descriptors.MolWt(m) <= 500
            and Descriptors.MolLogP(m) <= 5
            and Descriptors.NumHDonors(m) <= 5
            and Descriptors.NumHAcceptors(m) <= 10)

for s, m in zip(smiles_list, mols):
    if m:
        print(f"{s:40} -> {'проходит' if lipinski_ok(m) else 'НЕ проходит'}")


## Шаг 4. PAINS-фильтр

Проверка на «красные флаги» — фрагменты, часто дающие ложные результаты.

In [ ]:
from rdkit.Chem import FilterCatalog

catalog = FilterCatalog.FilterCatalog(
    FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
for s, m in zip(smiles_list, mols):
    if not m:
        continue
    match = catalog.GetFirstMatch(m)
    print(f"{s:40} -> PAINS: {'ЕСТЬ (' + match.GetDescription() + ')' if match else 'нет'}")


## Шаг 5. Таблица-отчёт и выводы

Соберите всё в одну таблицу и опишите: какие молекулы не прошли и почему.

In [ ]:
import pandas as pd

rows = []
for s, m in zip(smiles_list, mols):
    if m is None:
        rows.append({"SMILES": s, "читается": False})
        continue
    rows.append({
        "SMILES": s,
        "масса": round(Descriptors.MolWt(m), 1),
        "logP": round(Descriptors.MolLogP(m), 2),
        "Липински": "да" if lipinski_ok(m) else "нет",
    })
pd.DataFrame(rows)


**Выводы:**

- ...